# **Experiment Notebook**



---
## Setup Environment

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT1",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.


MessageError: Error: credential propagation was unsuccessful

---
## Student Information


In [ ]:
student_name = "Nonthawat Praisompong"
student_id = "25233750"

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

### 0.b Import Packages

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
import scipy.stats as stats

---
## A. Project Description


In [ ]:
business_objective = """
The objective of this project is to build predictive regression models (linear regression and KNN) that accurately predict the motor vehicle insurance premiums paid by customers.
By understanding the variables that most strongly influence net premiums, the company can personalise insurance offerings, minimise pricing risk, and maximise profitability.
These models will primarily serve and enhance product & service development and pricing teams' capability to design fair, competitive, and customer-specific policies.
Ultimately, the predictions will support better risk management, customer satisfaction, and sustainable business growth.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='business_objective', value=business_objective)

---
## B. Dataset Understanding (Global Interpretation)

In [ ]:
# put the path folder into the folder_path by using Path
folder_path = Path("/content/gdrive/MyDrive/36106/assignment/AT1/data")

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load training data
try:
  training_df = pd.read_csv(folder_path / "car_insurance_premium_training.csv")
  validation_df = pd.read_csv(folder_path / "car_insurance_premium_validation.csv")
  testing_df = pd.read_csv(folder_path / "car_insurance_premium_testing.csv")
except Exception as e:
  print(e)

In [ ]:
## Use function .set_option to display all dataset columns
pd.set_option('display.max_columns', None)

### B.1 Explore Training Set


> In this process, we will deploy EDA to explore, understand the customer characteris before model development.

> This analysis will contain with descriptive statistic and visualisation to uncover and check the completeness of the data.


Method applying for our analysis:
* Shape of the dataset
* Descriptive statistic
* Null and duplicate value
* The correctness of the data point (Is there any incorrect value within the dataset?)
* Visualisation: Box plot for category columns toward the target variable,
Histrogram for distribution and Scatter plot for relationship


In [ ]:
### Checking the shape of training df
print(training_df.shape)      # 40 features in total

In [ ]:
### Check the first 5 rows of the df
training_df.head()       # result: lapsed_date and vehicle_length has missing values

In [ ]:
### Checking the bottom 5 rows of the Training df.
training_df.tail()       # result: prefix and lapsed_date has missing values

In [ ]:
### Checking the completeness and correctness of the Training df (dtype, columns)
print(training_df.info())

## Overall, this df has 40 features which contain wiht object, int64  and float64 dtypes. (object: 19, int64: 16, float64: 5)
# However, some feature contains missing values.

In [ ]:
### How many features does each dtype contain?
training_df.dtypes.value_counts().to_frame('How many features')

In [ ]:
### Explore the descriptive statistics of numeric columns.
training_df.describe().round(2)

# Overall, seniority, current_policies_held and max_policies_held might have outliers due to outstanding max values
# Also these features: total_claims_cost_in_current_year, total_claims_number_in_current_year, total_claims_number_in_history	total_claims_number_ratio

In [ ]:
### Display statistics for object columns to observe mode and missing value.
training_df.describe(include = 'object')

# Trivial observation result:
# vehicle_fuel_type: Most customers use Diesel cars, which is surprise.
# distribution_channel: Mostly from the agent
# gender: unknown or unisex (u) is the most frequent.

In [ ]:
### What are the columns that have missing values?
print(training_df.isnull().sum()[lambda x: x > 0].to_frame('The number of the missing values')) # Using .isnull() function to find the missing values and .sum() to sum the value.

## Overall, there are 5 variables with missing value: first_name ,last_name, lapsed_date, vehicle_fuel_type, vehicle_length
#which lapsed_date has the highest number following prefix.

In [ ]:
### Display the duplicate low in the data frame.
dup = training_df.duplicated().sum()
print('Number of dup values:', dup)  # There is no Duplicate values in this data frame

In [ ]:
### Count the values of some interest Categorical variables in this training dataset
print(training_df['gender'].value_counts())  ## The gender distribution in this dataset is very similar, which they have similar number across all groups.
print(' ')
print(training_df['distribution_channel'].value_counts()) ## The distribution is quite balance between 2 groups. However, "00/01/1900" is incorrect values that we should dealing in cleaning process.
print(" ")
print(training_df['payment_method'].value_counts()) ## Indicating, majority of prefer annual payment (0 is at 20521).
print(" ")
print(training_df['policy_type'].value_counts()) # Passenger cars(3) has dominate as the most popular insurance.
print(" ")
print(training_df['second_driver'].value_counts()) ## Majority customers declare as the only one driver of the car.
print(' ')
print(training_df['vehicle_fuel_type'].value_counts())  ## Interestingly, Most cars are diesel car instead of Petrol
print(' ')
print(training_df['vehicle_doors'].value_counts()) ## Interestingly, 5-door cars have the highest number, which is outstanding from the other car types.
print(' ')
print(training_df['max_products_held'].value_counts()) ## The majority hold only 1 insurance for their vehicle.

In [ ]:
### Let's explore variables that I'm interested in or consider that it is might be related to the target variable (Net Premium Amount)

## Compared to each subgroup, how different of the amount of payment that each group pay for their premium
interest_vars = ['gender', 'distribution_channel', 'payment_method', 'policy_type', 'second_driver', 'vehicle_fuel_type', 'vehicle_doors', 'max_products_held']

f, axes = plt.subplots(3, 3, figsize=(20, 18))
axes = axes.flatten()    ### This line transforms the data matrixs or multiple arrays into a 1D array for doing for loop.
for i, v in enumerate(interest_vars):   ### If we do not use enumerate, SNS will plot variable in this same graph.
    sns.boxplot(data = training_df , x = v, y='net_premium_amount', ax = axes[i], palette='Set3', showmeans = True)
    axes[i].set_title(f'Net premium amouunt vs {v}')
    axes[i].set_xlabel(v)
    axes[i].set_ylabel('Net premium amount')
plt.tight_layout()
plt.show()

In [ ]:
## Distribution for numerical by using histrogram
numeric_df = training_df.select_dtypes(include = 'number')

fig, axes = plt.subplots(7, 3, figsize=(18, 21))
axes = axes.flatten()

for i, var in enumerate(numeric_df):
    sns.histplot(training_df[var], kde=True, bins=30, ax=axes[i], color='salmon')
    axes[i].set_title(f'Distribution of {var}')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
### Visual the relationship between continuous variable one on one by scatter plots.

## Net premium VS car price
## Firstly, we have to transform the values by using log for car price first
vehicle_value_log = np.log(training_df['vehicle_value'])

## Then visualise it.
plt.figure(figsize = (8, 5))
sns.scatterplot(x = vehicle_value_log , y = training_df['net_premium_amount'])
plt.title('Net premium vs log(car price)')
plt.xlabel('Car Price (Log)')
plt.ylabel('Net Premium amount')
plt.show()

## Net premium VS seniority
plt.figure(figsize = (8, 5))
sns.scatterplot(x = training_df['seniority'] , y = training_df['net_premium_amount'])
plt.title('Net premium vs seniority')
plt.xlabel('seniority')
plt.ylabel('Net Premium amount')
plt.show()

## Net premium VS current_policies_held
plt.figure(figsize = (8, 5))
sns.scatterplot(x = training_df['current_policies_held'] , y = training_df['net_premium_amount'])
plt.title('Net premium vs current_policies_held')
plt.xlabel('current_policies_held')
plt.ylabel('Net Premium amount')
plt.show()

## Net premium VS max_policies_held
plt.figure(figsize = (8, 5))
sns.scatterplot(x = training_df['max_policies_held'] , y = training_df['net_premium_amount'])
plt.title('Net premium vs max_policies_held')
plt.xlabel('max_policies_held')
plt.ylabel('Net Premium amount')
plt.show()


### In summary, these variables did not show the sign of the relationship toward the target variable.

In [ ]:
#### How claim related variables influence the premium price paid by customer?
## Net premium VS total_claims_cost_in_current_year
plt.figure(figsize = (8, 5))
sns.scatterplot(x = training_df['total_claims_cost_in_current_year'] , y = training_df['net_premium_amount'])
plt.title('Net premium vs total_claims_cost_in_current_year')
plt.xlabel('total_claims_cost_in_current_year')
plt.ylabel('Net Premium amount')
plt.show()

## Net premium VS total_claims_number_in_current_year
plt.figure(figsize = (8, 5))
sns.scatterplot(x = training_df['total_claims_number_in_current_year'] , y = training_df['net_premium_amount'])
plt.title('Net premium vs total_claims_number_in_current_year')
plt.xlabel('total_claims_number_in_current_year')
plt.ylabel('Net Premium amount')
plt.show()

## Net premium VS total_claims_number_in_history
plt.figure(figsize = (8, 5))
sns.scatterplot(x = training_df['total_claims_number_in_history'] , y = training_df['net_premium_amount'])
plt.title('Net premium vs total_claims_number_in_history')
plt.xlabel('total_claims_number_in_history')
plt.ylabel('Net Premium amount')
plt.show()

## Net premium VS total_claims_number_ratio
plt.figure(figsize = (8, 5))
sns.scatterplot(x = training_df['total_claims_number_ratio'] , y = training_df['net_premium_amount'])
plt.title('Net premium vs total_claims_number_ratio')
plt.xlabel('total_claims_number_ratio')
plt.ylabel('Net Premium amount')
plt.show()


### According to the result, these features does not seem having relationship with target variables

In [ ]:
### Visualise the Distribution of features that related to Claim History.

## total_claims_cost_in_current_year Distribution
plt.figure(figsize = (7, 5))
sns.histplot(training_df['total_claims_cost_in_current_year'])
plt.title('total_claims_cost_in_current_year Distribution')
plt.xlabel('total_claims_cost_in_current_year')
plt.ylabel('Frequency')
plt.show()


## total_claims_number_in_current_year Distribution
plt.figure(figsize = (7, 5))
sns.histplot(training_df['total_claims_number_in_current_year'])
plt.title('total_claims_number_in_current_year Distribution')
plt.xlabel('total_claims_number_in_current_year')
plt.ylabel('Frequency')
plt.show()


## total_claims_number_in_history Distribution
plt.figure(figsize = (7, 5))
sns.histplot(training_df['total_claims_number_in_history'])
plt.title('total_claims_number_in_history Distribution')
plt.xlabel('total_claims_number_in_history')
plt.ylabel('Frequency')
plt.show()


## total_claims_number_ratio Distribution
plt.figure(figsize = (7, 5))
sns.histplot(training_df['total_claims_number_ratio'])
plt.title('total_claims_number_ratio Distribution')
plt.xlabel('total_claims_number_ratio')
plt.ylabel('Frequency')
plt.show()


## lapsed_policies Distribution
plt.figure(figsize = (7, 5))
sns.histplot(training_df['lapsed_policies'])
plt.title('lapsed_policies Distribution')
plt.xlabel('lapsed_policies')
plt.ylabel('Frequency')
plt.show()


## matriculation_year Distribution
plt.figure(figsize = (7, 5))
sns.histplot(training_df['matriculation_year'])
plt.title('matriculation_year Distribution')
plt.xlabel('matriculation_year')
plt.ylabel('Frequency')
plt.show()

### As a result, all variables except matriculation_year have positive skews distribution (right skew).

In [ ]:
training_set_insights = """
Overall, this training dataset has 40 features and 32,136 rows containing 5 columns with missing values: prefix, last_name, lapsed_date, vehicle_fuel_type and vehicle_length.
In addition, some number columns have an outlier that we will decide later how do we going to deal wiht them. In the distribution_channel feature, has incorrect values shown as 00/01/1900 instead of 0 and 1.
To pique the curiosity. Most customers registered with AGENT and most of their cars are DIESEL. For gender, most of them have unisex gender.
Interestingly, the majority of insurance holders claim that only holder use or commutes the car (second driver). Most holders registered for passenger or personal car (policy_type). As expected, the mode of payment_method indicates annual payment.
Lastly, the distribution of tnet_premium_amount is between approximately 500 and 650.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='training_set_insights', value=training_set_insights)

### B.2 Explore Validation Set

> In this part, We will focus on the Validaton set.

To maintain validation and test data frame to be unseen, Light EDA will only be apply for observing the completeness of the data.


In this case, we are going to check missing and duplicate values, including incorrect value which determine the methodolog of data preprocessing.
In addition, the visualisation will not be applied in this EDA to maintain unseen.

In [ ]:
### Checking shape of validation data frame
check = validation_df.shape
print('There are', check[0], 'rows in this set',)
print('There are', check[1], 'columns in this set',)

## this validaton set has 10,700 rows with 40 columns in total.

In [ ]:
## Checking top 5 rows of validation df
validation_df.head()   # The columns with missing from validation set are the same as training dataset.

In [ ]:
## Checking the bottom rows of this df
validation_df.tail()

In [ ]:
### Checking the completeness and correctness of the Validation df, including the data type (dtype) of each column.
validation_df.info()

In [ ]:
## Checking the dtype and count.
validation_df.dtypes.value_counts()

In [ ]:
### Display descriptive statistic for numeric columns.
validation_df.describe().round(2)

In [ ]:
### How many null values in validation df?
print(validation_df.isnull().sum()[lambda x: x > 0].to_frame("What are the variables that contains null values and how many of them"))

## As the result, these are the same feature which has missing values like we have in training df

In [ ]:
### How many duplicate values in df?
validation_df[validation_df.duplicated()]   ## Good news, there is no duplication in this df. Actually, it is the same result as training set

In [ ]:
### Checking the incorrect values in distribution_channel
## I decide to use .value_counts() for checking the incorrect values. However, using this causes bias in feature engineering and makes the data not unseen.

print(validation_df['distribution_channel'].value_counts())  ## As a result, there is 00/01/1900 as incorrect value that we have fix it in data cleaning process

In [ ]:
# <Student to fill this section and then remove this comment>
validation_set_insights = """
Overall, the validation characteristics also have 40 columns but 10,700 rows.
It has the same issues and at the same columns that training set has consisting of missing values, incorrect values, outllier which we are able to use the same method for data preparation.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='validation_set_insights', value=validation_set_insights)

### B.3 Explore Testing Set

> It is time in testing set exploration.
> We will go with the same concept as validation which maintain this set to be unseen as much as possible by applying Light EDA

In this case, we are going to check missing and duplicate value including incorrect value.

In [ ]:
### Checking shape of validation testing data frame
testing_df.shape
## the size of the testing df is very similar (10700 in validation) which has 40 variables and 10,066 rows

In [ ]:
### Checking the first 5 rows
testing_df.head()  ### There is missing value in the same columns as trainig and validation.

In [ ]:
### Checking the last  5 rows
testing_df.tail()

In [ ]:
## Checking the correctness and data type of variables
testing_df.info()

In [ ]:
## Checking the dtype and count for the testing set.
testing_df.dtypes.value_counts()

In [ ]:
### How many null values in testing df?
print(testing_df.isnull().sum()[lambda x: x > 0].to_frame("What are the variables that contains null values and how many of them"))

In [ ]:
### Check the incorrect values in distribution channel
print(training_df['distribution_channel'].value_counts().to_frame('Unique values'))
## As the result, there 00/01/1900 is still in testing df which we have to deal with it in preparation process.

In [ ]:
### Checking the duplicate values in testing set
validation_df.duplicated().value_counts()  ## There is no duplicate values in Testing df

In [ ]:
# <Student to fill this section and then remove this comment>
testing_set_insights = """
In summary, this set has 10,666 rows and 40 columns. Ultimately, it has the same issue as the training and validation set which we will fix in the preparation notebook.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='testing_set_insights', value=testing_set_insights)

---
## C. Feature Understanding (Local Interpretation)

### C.1 Explore Target Variable

> Save the name of column used as the target variable and call it `target_name`

In [ ]:
## We will train our model to predict variables which influnce "net_premium_amount"
## set the target_name
target_name = training_df['net_premium_amount']

In [ ]:
### Target variable characteristic?

print(f"Data type: {target_name.dtype}")
print(f"Data points: {target_name.shape}")
print(''' ''')
print(target_name.describe().round(2))      # the distribution are of net_premium_amount is between 460 to 676.

In [ ]:
### Is there any missing values?
print(target_name.isnull().sum()) ## there is no missing values

In [ ]:
### Finding the distribution of the Target variable by visualising
plt.figure(figsize = (6, 4))
sns.histplot(target_name)
plt.title('Net Premium Amount Distribution')
plt.xlabel('Net Premium Amount')
plt.ylabel('Frequency')
plt.show()

## This chart has the right skew distribution with the majority between early 500 and more than 650.
# this mean, custoer tend to

In [ ]:
### Using log transformation to observe the real distribution
target_name_log = np.log(target_name)

### Visualise it!
plt.figure(figsize = (6, 4))
sns.histplot(target_name_log)
plt.title('Log Net Premium amount Distribution')
plt.xlabel('Log Net Premium amount')
plt.ylabel('Frequency')
plt.show()

## As a result, the distribution is still the same (right skwer)

In [ ]:
### QQ plot for visualisation distribution
plt.figure(figsize=(4, 3))
stats.probplot(target_name, dist="norm", plot=plt)
plt.title('QQ Plot of Net Premium amount')
plt.show()

## THis chart emphasise that the chart has the right skew distribution.

In [ ]:
### Checking the outlier by box plot visualising
plt.figure(figsize = (4, 4))
sns.boxplot(target_name)
plt.title('Summary of Net Premium amount')
plt.xlabel('Net Premium amount')
plt.show()

## there is not outlier in net_premium_amount

In [ ]:
# <Student to fill this section and then remove this comment>
target_insights = """
Overall, the target variables show a slight left skew. However, the majority of data have a positive distribution, indicating the spread is on the high values. On the other hand, our customers tend to pay for higher premiums if insurance price and products match their interest.
The result from box plot show no outlier within this target feature.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='target_insights', value=target_insights)

### C.2 Explore Feature of Interest `<seniority>`

> My first interest variable is senority which indicating total number of years that insurance holder has contracted with the insurance company.

In [ ]:
### Senority is the variable that I think it is directly assciate with target variable.
seniority = training_df['seniority']

In [ ]:
### Checking the seniority characteristic

print(f"Data type: {seniority.dtype}")
print(f"Data points: {seniority.shape}")
print(''' ''')
print(seniority.describe().round(2))   ## max value at 40 while 50% and 75% are 4 and 7 which indicate outlier within this feature.

In [ ]:
### Is there any missing values?
print(seniority.isnull().sum())  ## Also no missing values

In [ ]:
### Finding the distribution of the seniority by visualising
plt.figure(figsize = (6, 3))
sns.histplot(seniority)
plt.title('seniority Distribution')
plt.xlabel('seniority')
plt.ylabel('Frequency')
plt.show()

## This chart shows a right-skewed distribution, indicating that the majority have joined or registered for a contract around 1 to 5 years ago.
# In addition, long tails on the right determine few customers has longer contracted up to 40 year.

In [ ]:
### Using log transformation to observe the real distribution
seniority_log = np.log(seniority)

### Visualise it!
plt.figure(figsize = (6, 3))
sns.histplot(seniority_log)
plt.title('Log seniority Distribution')
plt.xlabel('Log seniority')
plt.ylabel('Frequency')
plt.show()

## After applying log transformation, seniority still how the right skew distribution or right skew.

In [ ]:
### How about the relationship with target variable?
plt.figure(figsize = (6, 3))
sns.scatterplot(x = seniority , y = target_name)
plt.title('Seniority vs Net premium amount')
plt.xlabel('seniority')
plt.ylabel('net premium amount')
plt.show()

### There is not sign of relationhship between these 2 variables.

In [ ]:
### Checking the outlier by box plot visualising
plt.figure(figsize = (7, 4))
sns.boxplot(x = seniority, width = 1, gap= 0.5)
plt.title('Summary of seniority')
plt.xlabel('seniority')
plt.show()

## This chart show there are outlier after 15 years

In [ ]:
# <Student to fill this section and then remove this comment>
feature_1_insights = """
The seniority feature has the right skew distribution, with the majority lying approximately between 1 and 3. The observations above 15 are considered as outliers which are identified by the box plot.
The correlation between the target feature. The scatter plot shows no sign of a relationship.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### C.3 Explore Feature of Interest `<vehicle_value>`

> My second interest variable is Vehicle value, which I am curious about how it influence target variable? What is thier distribution?

In [ ]:
### vehicle_value is our third feature interest
vehicle_value = training_df['vehicle_value']

In [ ]:
### Checking the descriptive statistic

print(f"Data type: {vehicle_value.dtype}")
print(f"Data points: {vehicle_value.shape}")
print(''' ''')
print(vehicle_value.describe().round(2))  # This variable contain outlier when compare value of 50, 75 to max. However, it is understandable due to car and motocycle has various price level.

In [ ]:
### Finding the distribution of the vehicle_value by visualising
plt.figure(figsize = (6, 3))
sns.histplot(vehicle_value)
plt.title('vehicle value Distribution')
plt.xlabel('vehicle value')
plt.ylabel('Frequency')
plt.show()

## It has the right skew distribution that data lie mostly between 10,000 to 25,000

In [ ]:
### QQ plot for visualisation distribution
plt.figure(figsize=(4, 3))
stats.probplot(vehicle_value, dist="norm", plot=plt)
plt.title('QQ Plot of vehicle value')
plt.show()    # this confirm the right skew distribution.

In [ ]:
### Log transform and visualise it
vehicle_value_log = np.log(vehicle_value)


plt.figure(figsize = (6, 3))
sns.histplot(vehicle_value_log)
plt.title('Log vehicle value Distribution')
plt.xlabel('log vehicle value')
plt.ylabel('Frequency')
plt.show()

## After we normalise it, the distribution has been change to left skew which majority values lie between 9.5 to 10.5. It show the real distribution of this feature.

In [ ]:
### How about the relationship with target variable?
plt.figure(figsize = (6, 3))
sns.scatterplot(x= vehicle_value , y= target_name)
plt.title('vehicle_value vs Net premium amount')
plt.xlabel('vehicle_value')
plt.ylabel('net premium amount')
plt.show()

### There is not sign of relationhship between these 2 variables.

In [ ]:
### Checking the outlier by box plot visualising
plt.figure(figsize = (7, 4))
sns.boxplot(x = vehicle_value, width = 1, gap= 0.5)
plt.title('Summary of vehicle value')
plt.xlabel('vehicle value')
plt.show()

## this chat show the outlier of this variable

In [ ]:
# <Student to fill this section and then remove this comment>

feature_2_insights = """
Fron descriptive statistic, the different between 75% and max show sign of ourtlier. It show the right skew distribution which the majority lie between appoximately 15,000 to 25,000.
For outlier investigation, box plot show huge amount of outlier near zero and 35,000 which understandable from the various price of car.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### C.4 Explore Feature of Interest `<vehicle_cylinder>`

> My third Feature of Interest is vehicle_cylinder or CC which might directly influence the target variable

In [ ]:
### vehicle_cylinder is our fourth feature interest
vehicle_cylinder = training_df['vehicle_cylinder']

In [ ]:
### Checking the descriptive statistic

print(f"Data type: {vehicle_cylinder.dtype}")
print(f"Data points: {vehicle_cylinder.shape}")
print(''' ''')
print(vehicle_cylinder.describe().round(2))   # The different of cc are various due to car types (sport, luxury, super car, etc.)

In [ ]:
### Finding the distribution of the vehicle_cylinder by visualising
plt.figure(figsize = (6, 3))
sns.histplot(vehicle_cylinder)
plt.title('vehicle cylinder Distribution')
plt.xlabel('vehicle cylinder')
plt.ylabel('Frequency')
plt.show()

## This chart show multimodal distribution which peak around 1,500–2,000 cc

In [ ]:
### QQ plot for visualisation distribution
plt.figure(figsize=(4, 3))
stats.probplot(vehicle_cylinder, dist="norm", plot=plt)
plt.title('QQ Plot of vehicle cylinder')
plt.show()

## From this QQ plot obvious indicated this feature has the right skew distribution.

In [ ]:
### Log transform and visualise it
vehicle_cylinder_log = np.log(vehicle_cylinder)


plt.figure(figsize = (6, 3))
sns.histplot(vehicle_cylinder_log)
plt.title('Log vehicle cylinder Distribution')
plt.xlabel('log vehicle cylinder')
plt.ylabel('Frequency')
plt.show()

## After log transformation, the chart show it true distribution which is left skew.

In [ ]:
### How about the relationship with target variable?
plt.figure(figsize = (6, 3))
sns.scatterplot(x= vehicle_cylinder , y= target_name)
plt.title('vehicle_cylinder vs Net premium amount')
plt.xlabel('vehicle_cylinder')
plt.ylabel('net premium amount')
plt.show()

### There is not sign of relationhship between these 2 variables.

In [ ]:
### Checking the outlier by box plot visualising
plt.figure(figsize = (7, 4))
sns.boxplot(x = vehicle_cylinder, width = 1, gap= 0.5)
plt.title('Summary of vehicle cylinder')
plt.xlabel('vehicle cylinder')
plt.show()

## There is the outlier in this engine size variable

In [ ]:
# <Student to fill this section and then remove this comment>

feature_4_insights = """
Overall, thie variable has multimodal distribution which peak around 1,500–2,000 cc, the scatter plot show no sign of relationship with target variables. However, box plot show huge amout of outliers which is understandable.
Our insurance has many insurance type for various car type the distribution and boxplot outlier are understandable in reality due to this reason.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_3_insights', value=feature_4_insights)

### C.5 Explore Feature of Interest `<vehicle_horsepower>`

> My last interest feature is vehicle_horsepower

In [ ]:
### vehicle_cylinder is our fifth feature interest
vehicle_horsepower = training_df['vehicle_horsepower']

In [ ]:
### Checking the descriptive statistic

print(f"Data type: {vehicle_horsepower.dtype}")
print(f"Data points: {vehicle_horsepower.shape}")
print(''' ''')
print(vehicle_horsepower.describe().round(2))   ## Min horse power  = 0 indicating the incorrect values.

In [ ]:
### Finding the distribution of the vehicle_horsepower by visualising
plt.figure(figsize = (6, 3))
sns.histplot(vehicle_horsepower)
plt.title('vehicle horsepower Distribution')
plt.xlabel('vehicle horsepower')
plt.ylabel('Frequency')
plt.show()

## This chart has rigt skew distribution and the distribution of majoriy fall aproximately between 60 to 120 with long tai on the right.

In [ ]:
### QQ plot for visualisation distribution
plt.figure(figsize=(4, 3))
stats.probplot(vehicle_horsepower, dist="norm", plot=plt)
plt.title('QQ Plot of vehicle horsepower')
plt.show()

In [ ]:
### Log transform and visualise it
vehicle_horsepower_log = np.log(vehicle_horsepower)


plt.figure(figsize = (6, 3))
sns.histplot(vehicle_horsepower_log)
plt.title('Log vehicle horsepower Distribution')
plt.xlabel('log vehicle horsepower')
plt.ylabel('Frequency')
plt.show()

In [ ]:
### How about the relationship with target variable?
plt.figure(figsize = (6, 3))
sns.scatterplot(x= vehicle_horsepower , y= target_name)
plt.title('vehicle_horsepower vs Net premium amount')
plt.xlabel('vehicle_horsepower')
plt.ylabel('net premium amount')
plt.show()

### There is not sign of relationhship between these 2 variables.

In [ ]:
### Checking the outlier by box plot visualising
plt.figure(figsize = (7, 4))
sns.boxplot(x = vehicle_horsepower, width = 1, gap= 0.5)
plt.title('Summary of vehicle_horsepower')
plt.xlabel('vehicle_horsepower')
plt.show()

## There is the outlier in this engine size variable

In [ ]:
# <Student to fill this section and then remove this comment>

feature_n_insights = """
In summary, this feature has right skew distribution and the distribution of majoriy fall aproximately between 60 to 120 with long tail on the right. The scatter plot display lots of outlier indicating most car within this dataset are commute car.
Interestingly there is outliers lie arourd 0 to 25 indicating there is various of car or motocycle type which the data point which have horse power = 0 indating the incorrect values.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)